# Capstone B: a breach across teams

A breach at a payroll company that crosses every team: a password spray on the VPN, a scan and a script with secrets on a file share, a hop to an application server, salary files copied out on-premises and from the cloud. No hints. Rebuild it, scope it, contain it, file and notify on time as a data processor, leave new detections behind, and brief the board.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `captured/backup-config.ps1.txt`, `certin-indigo.json`, `clients.csv`, `cloud/audit.json`, `cloud/egress-alert.txt`, `cloud/policies.json`, `eventlogs/pay-app01-Security.jsonl`, `identity/users.csv`, `README.txt`, `vpn.log`, `zeek/conn.log`, `zeek/smb_files.log`.

## 1. Brief, custody, plan

Read the README and the alert, hash everything, and write a one-paragraph plan: what you'll look at, in what order, and what you need to know first for the clocks.

- [ ] Every file hashed
- [ ] The detection time written down, in UTC
- [ ] A plan before you start

?> How many evidence files are there? :: sha256:6b51d431df5d7f141cbececcf79edf3dd861c3b4069f0b11661a3eefacbba918
?> At what time (UTC, HH:MM) did Indigo notice the incident? :: sha256:f8cf4a738b5eb2123102adb0c96db3038838534470fd8ec3dccc9a44173064d0

In [ ]:
%%bash
cd ~/case
find . -type f | sort
echo ---
find . -type f -exec sha256sum {} + | sort -k2
echo ---
cat cloud/egress-alert.txt

## 2. How they got in

Find the entry point in the VPN log and the directory export.

- [ ] What pattern do the failures show?
- [ ] Which account got in, and why did that one work?
- [ ] What should have happened to that account months ago?

?> How many accounts were tried from 198.51.100.150? :: sha256:4ec9599fc203d176a301536c2e091a19bc852759b255bd6818810a42c5fed14a
?> Which account got in? :: sha256:6c5edee16a932fad272e0fd6ff1b75efca59a7b5e39f6339a87e6fd709340d3d

In [ ]:
%%bash
cd ~/case
grep -c 'src=198.51.100.150' vpn.log
grep 'src=198.51.100.150' vpn.log | grep success
echo ---
grep intern identity/users.csv

## 3. Across the network

Use Zeek to see what the VPN address did inside.

- [ ] How many addresses did it probe on 445? Which answered?
- [ ] Which share and file did it open on files01?
- [ ] What was in that file? Why does that change everything?

?> How many different addresses did 10.80.9.33 probe on port 445? :: sha256:71ee45a3c0db9a9865f7313dd3372cf60dca6479d46261f3542eb9346e4a04d6
?> Which file did the attacker open on the IT$ share? (file name) :: sha256:21e383631865c46c3a9599c0be3d5bc994c4a90aaac7750a8b3f2d00336cec4b

In [ ]:
%%bash
cd ~/case/zeek
grep -v '^#' conn.log | awk -F'\t' '$3=="10.80.9.33" && $6=="445" {print $5, $12}' | sort -u | awk '{print $2}' | sort | uniq -c
echo ---
grep -v '^#' smb_files.log | awk -F'\t' '{print $5, $6, $7}'

## 4. On the application server

Read pay-app01's Security log.

```tool events
{"file": "eventlogs/pay-app01-Security.jsonl", "q": "localadmin", "n": 7, "blank": ["ids", "src", "q"]}
```

- [ ] Who logged on, how, from where?
- [ ] What did they run, in order?
- [ ] Where did the data go?
- [ ] What did they try to hide?

?> Which domain received the uploaded archive? :: sha256:94bbf9e989de4a107efceb5617c6d85f5cde06080b92a4eda6c6123ba9fe2bdb
?> Which event ID shows the log was cleared? :: sha256:277375b99e186c72ac38ac47b03199038342fe0389be8765476fa2be0c5b5649

## 5. In the cloud

Follow the backup key through the audit trail.

- [ ] What did the key do, from where, and what was denied?
- [ ] Which clients' files, which months?

?> How many objects did the attacker download from the bucket? :: sha256:6b51d431df5d7f141cbececcf79edf3dd861c3b4069f0b11661a3eefacbba918

In [ ]:
%%bash
cd ~/case/cloud
grep -c GetObject audit.json
grep -o 'payroll-exports/[a-z-]*/' audit.json | sort | uniq -c

## 6. One timeline

Put the VPN, the server and the cloud on one clock. Star the key events and download the timeline for the report.

```tool timeline
{"keys": ["vpn.log#19@1827457119", "eventlogs/pay-app01-Security.jsonl#3@1827461109", "cloud/audit.json#4@1827462003", "eventlogs/pay-app01-Security.jsonl#6@1827462700", "eventlogs/pay-app01-Security.jsonl#7@1827463948", "eventlogs/pay-app01-Security.jsonl#9@1827464520"], "n": 6, "blank": ["keys"]}
```

- [ ] Entry, movement, theft (two routes), cover-up, detection
- [ ] The gap between the theft and detection

?> How many minutes passed between the VPN login and the first cloud download? (whole minutes) :: sha256:bbb965ab0c80d6538cf2184babad2a564a010376712012bd07b0af92dcd3097d

## 7. Scope: whose data, how much

Work out exactly whose data was taken, and who has to know.

- [ ] People affected, by client
- [ ] Which client is itself regulated, and what that means for its own clocks
- [ ] Who has employees in the EU?

?> How many people's payroll records were in the stolen exports? :: sha256:9060ce5653093cd9bff480cb01ff0e3a790e6f545e214a3dd85841daabac7848

In [ ]:
%%bash
cd ~/case
cat clients.csv
awk -F, 'NR>1 {s+=$2} END {print s, "people"}' clients.csv

## 8. Contain and fix

Order the containment, then prove the cloud fix with a replay.

```tool cloudiam
{"pol": "{\n \"account\": \"indigo-prod\",\n \"identities\": [\n  {\n   \"name\": \"payroll-backup\",\n   \"type\": \"user\",\n   \"note\": \"key in backup-config.ps1; nightly upload of D:\\\\exports from pay-app01 (office egress 203.0.113.88)\",\n   \"policies\": [\n    {\n     \"name\": \"upload-exports-only\",\n     \"statements\": [\n      {\n       \"effect\": \"Allow\",\n       \"actions\": [\n        \"PutObject\"\n       ],\n       \"resources\": [\n        \"bucket/payroll-exports/*\"\n       ],\n       \"conditions\": {\n        \"source_ip\": [\n         \"203.0.113.88/32\"\n        ]\n       }\n      }\n     ]\n    }\n   ]\n  },\n  {\n   \"name\": \"c.iyer\",\n   \"type\": \"user\",\n   \"note\": \"cloud administrator\",\n   \"policies\": [\n    {\n     \"name\": \"admin\",\n     \"statements\": [\n      {\n       \"effect\": \"Allow\",\n       \"actions\": [\n        \"*\"\n       ],\n       \"resources\": [\n        \"*\"\n       ]\n      }\n     ]\n    }\n   ]\n  }\n ],\n \"org_guardrails\": [\n  {\n   \"name\": \"protect-audit-trail\",\n   \"statements\": [\n    {\n     \"effect\": \"Deny\",\n     \"actions\": [\n      \"StopLogging\",\n      \"DeleteTrail\"\n     ],\n     \"resources\": [\n      \"*\"\n     ]\n    }\n   ]\n  },\n  {\n   \"name\": \"reads-of-client-data-need-mfa\",\n   \"statements\": [\n    {\n     \"effect\": \"Deny\",\n     \"actions\": [\n      \"GetObject\",\n      \"ListObjects\"\n     ],\n     \"resources\": [\n      \"bucket/payroll-exports*\"\n     ],\n     \"conditions\": {\n      \"mfa\": false\n     }\n    }\n   ]\n  }\n ],\n \"buckets\": [\n  {\n   \"name\": \"payroll-exports\",\n   \"public_read\": false,\n   \"encrypted\": true,\n   \"versioning\": true,\n   \"access_logging\": true,\n   \"note\": \"clients' payroll exports\"\n  }\n ]\n}\n", "n": 0, "broken": 0, "files": ["cloud/policies.json"], "blank": ["pol"]}
```

| Order | Action |
|---|---|
| 1 | Cut the VPN session; disable intern.payroll; block 198.51.100.150 and 203.0.113.230 |
| 2 | Revoke the backup key; reset localadmin on every server (unique passwords per server) |
| 3 | Isolate pay-app01; keep its disk image |
| 4 | Remove secrets from the IT$ scripts; restrict the share to IT |
| 5 | MFA on every VPN account; disable accounts on leaving day |
| 6 | Backup key: upload only, only from the office; reads of client data need MFA |

?> After the policy change, how many attacker actions are still allowed? :: sha256:5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9

## 9. File and notify

File with CERT-In, answer the inbox, and write the notification order. Remember who is the processor and who are the fiduciaries.

```tool certin
{"file": "certin-indigo.json", "form": {"name": "Chetan Iyer", "role": "Head of IT and security", "email": "security@indigoledger.example", "phone": "+91 20 6600 4000", "org": "Indigo Ledger Pvt Ltd", "sector": "IT services (payroll processing for client companies)", "occurred": "2027-11-29 02:58", "detected": "2027-11-29 05:35", "type": "Data breach", "systems": "VPN vpn01 (account intern.payroll); file server files01 (IT$ share); application server pay-app01 (10.80.1.40, Windows Server); cloud storage bucket payroll-exports.", "symptoms": "Password spray on the VPN from 198.51.100.150; one former intern's account without MFA got in; a script on a file share held a local admin password and a cloud key; payroll export files of four client companies (2,505 employees: names, salaries, bank accounts, PAN) were copied from pay-app01 and from cloud storage.", "indicators": "198.51.100.150; 203.0.113.230 and upload.drop-files.example (exfiltration); key SCKA9F2LQ7MX4TB1; C:\\Users\\Public\\t.ps1 and e.zip.", "actions": "VPN session cut 06:02; intern.payroll disabled; cloud key revoked; localadmin password reset on all servers; pay-app01 isolated; logs preserved.", "personal": "yes", "filed": "2027-11-29 09:10"}, "n": 0, "blank": ["form"]}
```

| # | Who | Why | Due (UTC) | Owner |
|---|---|---|---|---|
| 1 | CERT-In | Six hours from noticing; Indigo is a service provider | 29 Nov 11:35 | Head of IT and security |
| 2 | Each client (data fiduciary) | They must notify the Board and their employees; Indigo processes for them | At once, by 07:00 | CEO and account managers |
| 3 | Coral Small Finance Bank, first | It's RBI-regulated with its own six-hour clock from when it learns | At once | Head of IT and security |
| 4 | Clients' notices to the Data Protection Board and their employees | Their duty; Indigo supplies facts and a draft notice | Without delay | Each client's DPO, supported by Indigo |
| 5 | Fjord Design's EU parent | 12 EU staff: GDPR, 72 hours, their decision | Today | Account manager |
| 6 | Cyber insurer | Policy terms | Today | CFO |

?> Who must tell the affected employees: Indigo or its clients? (one word) :: sha256:8f4d9088121acfab37f4a5875ef93c8a89e1489f1fc3b4367a1ce950a44f23e8

## 10. Leave detections behind

Two rules that would have caught this hours earlier: one for the spray, one for the local admin logon.

```tool siem
{"q": "source=*vpn.log result=failure | stats count by src", "n": 1, "files": ["vpn.log"], "blank": ["q"]}
```

```tool sigma
{"rule": "title: Network logon by a local administrator account\nid: 4c7e1a20-5b3f-4d8e-8f1a-2c0d00001201\nstatus: experimental\ndescription: Local admin accounts should never log on over the network; with LAPS each server's password is different. A network logon by one usually means a reused password.\nlogsource:\n  product: windows\n  service: security\ndetection:\n  selection:\n    EventID: 4624\n    LogonType: '3'\n    TargetUserName: 'localadmin'\n  condition: selection\nfalsepositives:\n  - Break-glass use during an outage; record it in the change log\nlevel: high\ntags:\n  - attack.lateral_movement\n  - attack.t1078.003\n", "n": 1, "ls": true, "blank": ["rule"]}
```

- [ ] The spray search finds one source with many failures
- [ ] The Sigma rule matches the attacker's logon and nothing normal
- [ ] For each, the time it would have fired, against the time Indigo noticed

?> How many failed VPN logins came from the spraying address? :: sha256:4523540f1504cd17100c4835e85b7eefd49911580f8efff0599a8f283be6b9e3

## 11. Brief the board, report to the team

Two documents: a one-page board briefing and the full technical report.

```tool report
{"tpl": "escalation", "title": "Note to the board: client payroll data stolen, 29 November 2027", "text": "# Note to the board: client payroll data stolen, 29 November 2027\n\n## The ask\n\nApprove today: (1) we tell all four clients now and support their notices; (2) MFA on every remote login and same-day removal of leavers' access, starting this week; (3) an outside firm to confirm the scope.\n\n## Severity and why\n\nCritical. Salary, bank and PAN details of 2,505 employees of four client companies were copied by an outsider. This is our clients' data, entrusted to us.\n\n## What we know\n\n- An outsider guessed the password of a former intern's remote-access account that was never closed and had no second factor.\n- A script on our file share held a server password and a cloud key. With them, the outsider copied three months of payroll files for four clients, by two routes.\n- We noticed at 05:35 UTC; access was cut at 06:02; CERT-In was told at 09:10, inside six hours.\n\n## What we don't know\n\nWho did it, and whether the data has been published or sold. We have no sign of money being moved.\n\n## Impact so far\n\nFour clients and 2,505 people at risk of fraud. One client is a bank with its own regulator. Expect contract claims and questions from clients' regulators.\n\n## What I've done\n\nAccount disabled, key revoked, server passwords reset, server isolated, logs preserved, CERT-In filed, clients being called now.\n\n## Evidence\n\nTechnical report and timeline attached; evidence hashed and held under chain of custody.\n", "n": 7, "blank": ["text"]}
```

```tool report
{"tpl": "incident", "title": "Incident report: Indigo Ledger, clients' payroll data copied, 29 November 2027", "text": "# Incident report: Indigo Ledger, clients' payroll data copied, 29 November 2027\n\n## Summary\n\nA password spray on the VPN reached a former intern's account without MFA. A script on a file share gave the attacker a local admin password and a cloud key. Payroll exports for four clients (2,505 people) were copied from pay-app01 and from cloud storage.\n\n## Timeline\n\n02:51 to 03:01 VPN spray from 198.51.100.150 (18 accounts); 02:58:39 intern.payroll succeeds (no MFA). 03:15 SMB sweep of 10.80.2.0/24 and four servers. 03:40 backup-config.ps1 read on files01 IT$. 04:05 localadmin NTLM logon to pay-app01. 04:13 t.ps1 runs. 04:20 to 04:24 cloud key reads 12 objects from payroll-exports; 04:30 PutBucketPolicy denied. 04:31 exports zipped; 04:52 uploaded to upload.drop-files.example (182 MB to 203.0.113.230). 05:01 files deleted; 05:02 Security log cleared (1102). 05:20 egress alert; 05:35 acknowledged; 06:02 VPN session cut.\n\n## Scope\n\nData: payroll exports, September to November 2027, for Lotus Motors, Pine Textiles, Coral Small Finance Bank and Fjord Design India. Systems: vpn01, files01 (IT$), pay-app01, bucket payroll-exports. Not seen: other servers' logons, the database itself, other buckets.\n\n## Impact\n\n2,505 people's names, salaries, bank accounts and PAN. Fraud risk to them; contract and regulatory exposure for Indigo and its clients.\n\n## Root cause\n\nA leaver's account left active without MFA; secrets stored in a script on a readable share; one local admin password on every server; a cloud key that could read everything.\n\n## Containment, eradication and recovery\n\nAs ordered in the notebook: session cut, account disabled, addresses blocked, key revoked, unique local admin passwords (LAPS), pay-app01 isolated and rebuilt, secrets removed from scripts, MFA on all VPN accounts, least-privilege backup key with an MFA guardrail on reads.\n\n## Who was told, and when\n\nCERT-In 09:10 UTC; Coral Small Finance Bank 06:40; the other three clients by 07:00; insurer 08:00.\n\n## Indicators\n\n198.51.100.150; 203.0.113.230; upload.drop-files.example; key SCKA9F2LQ7MX4TB1; C:\\Users\\Public\\t.ps1, e.zip.\n\n## Lessons learned\n\nLeavers' access must end on their last day. Secrets belong in a vault, not a script. MFA on every remote login. The spray was visible three hours before anyone looked.\n", "n": 9, "blank": ["text"]}
```

- [ ] The board note asks for decisions first
- [ ] The technical report matches your timeline exactly

## 12. Hand in

Hand in this notebook. It carries every answer, tool result and report from the steps above.

**Rubric**

| Criterion | Points |
|---|---|
| Entry, movement and theft reconstructed across VPN, network, Windows and cloud, with times | 6 |
| Scope: which data, whose, and how many people, with the counts shown | 4 |
| Containment and fixes in the right order; the cloud policy at 0 attacker actions and 0 broken | 4 |
| CERT-In filed on time, and the notification order right for a processor and its clients | 5 |
| New detections tested on the evidence | 3 |
| Executive briefing a board can decide from | 4 |
| Technical report complete and consistent with the timeline | 4 |

- [ ] Every file hashed
- [ ] Timeline starred and downloaded
- [ ] Cloud policy at 0 and 0
- [ ] CERT-In filed with no problems
- [ ] Both detections tested
- [ ] Board note and technical report written
- [ ] Press Hand in